# Master the HAVING Clause

This notebook focuses exclusively on **HAVING** — the clause that filters *groups* after aggregation.

## Core idea

| Clause | When it runs | What it filters | Can use aggregates? |
|--------|--------------|-----------------|---------------------|
| `WHERE` | **Before** grouping | Individual rows | No |
| `HAVING` | **After** grouping | Groups | **Yes** |

```sql
SELECT   ...
FROM     ...
WHERE    <row conditions>     -- filters rows first
GROUP BY ...
HAVING   <group conditions>   -- filters groups second
ORDER BY ...
```

## Why HAVING exists

After `GROUP BY`, each group is represented by one row of aggregate values.  
`WHERE` can no longer see the original rows, so we need a different filter: **HAVING**.

## Common patterns you will practice

1. Simple threshold (`HAVING COUNT(*) > 5`)
2. Aggregate comparison (`HAVING SUM(Total) > 100`)
3. Multiple conditions with `AND` / `OR`
4. Mixing `WHERE` + `HAVING` correctly
5. Using aliases in `HAVING` (SQLite allows it)
6. `HAVING` without `GROUP BY` (rare but legal — treats the whole result as one group)
7. Subtle mistakes (putting row filters in HAVING, forgetting aggregates, etc.)

## Schema reminder

```
customers      → Country, SupportRepId, CustomerId, ...
invoices       → BillingCountry, Total, InvoiceDate, CustomerId
tracks + genres → GenreId, Name, UnitPrice, Milliseconds
employees      → Title, HireDate, ...
```


## Exercise 1 – Basic HAVING threshold

**Question:** Which countries have **more than 4** customers?

### Instructions
- Group customers by `Country`
- Keep only groups with more than 4 customers
- Show country and customer count
- Order by count descending


**Hints**
- `HAVING COUNT(*) > 4`
- Remember: `WHERE` cannot be used for this because the count only exists after grouping.


## Exercise 2 – HAVING with SUM

**Question:** Which billing countries generated **more than $150** in total revenue?

### Instructions
- From `invoices`
- Group by `BillingCountry`
- Filter groups with `SUM(Total) > 150`
- Return country and revenue, ordered by revenue descending


**Hints**
- `HAVING SUM(Total) > 150`
- You may alias the sum and still reference the original expression in HAVING (both work in SQLite).


## Exercise 3 – Multiple conditions (AND)

**Question:** Find billing countries that have **at least 10 invoices** **and** a maximum invoice total **greater than 15**.

### Instructions
- Group by `BillingCountry`
- Use `HAVING` with two conditions joined by `AND`
- Show country, invoice count, and max total
- Order by max total descending


**Hints**
```sql
HAVING COUNT(*) >= 10 AND MAX(Total) > 15
```


## Exercise 4 – OR conditions in HAVING

**Question:** List genres that either:
- have an average unit price **greater than 0.99**, **or**
- contain **more than 300** tracks

### Instructions
- Join `tracks` and `genres`
- Group by genre name
- Use `HAVING` with `OR`
- Return genre, track count, and average price
- Order by track count descending


**Hints**
```sql
HAVING AVG(t.UnitPrice) > 0.99 OR COUNT(*) > 300
```


## Exercise 5 – WHERE + HAVING together (the classic pattern)

**Question:** Looking **only at invoices from 2010 and later**, which countries have an average invoice total **greater than 6**?

### Instructions
1. Filter rows first with `WHERE` (year ≥ 2010)
2. Group by `BillingCountry`
3. Keep groups whose average total > 6 with `HAVING`
4. Order by average descending

Use `strftime('%Y', InvoiceDate)` to extract the year.


**Hints**
- `WHERE strftime('%Y', InvoiceDate) >= '2010'`
- `HAVING AVG(Total) > 6`
- Putting the year filter in HAVING would be incorrect (and less efficient) because the year is a row-level attribute.


## Exercise 6 – Alias in HAVING (SQLite convenience)

**Question:** Which support representatives manage **more than 18** customers?

### Instructions
- From `customers`
- Group by `SupportRepId`
- Give the count an alias `CustomerCount`
- Filter with `HAVING CustomerCount > 18` (using the alias)
- Order by customer count descending


**Hints**
- SQLite allows referencing a SELECT alias inside HAVING.
- Most other databases require you to repeat the aggregate expression instead of the alias.


## Exercise 7 – HAVING without GROUP BY

**Question:** Is the overall average invoice total greater than 5.5?  
Return the average only if the condition is true; otherwise return no rows.

### Instructions
- Calculate `AVG(Total)` from `invoices`
- Use `HAVING AVG(Total) > 5.5` **without** a `GROUP BY`
- This treats the entire table as a single group


**Hints**
- When there is no `GROUP BY`, the whole result set is one implicit group.
- HAVING can still filter that single group.
- This pattern is uncommon but useful for “return the aggregate only if it meets a condition”.


## Exercise 8 – Complex multi-aggregate HAVING

**Question:** Find support reps whose:
- total revenue is **greater than 700**, **and**
- average invoice value is **greater than 5.5**

### Instructions
Join `customers` and `invoices` on `CustomerId`.

Return:
- SupportRepId
- Total revenue
- Average invoice
- Number of invoices

Filter with a compound `HAVING`, order by total revenue descending.


**Hints**
```sql
HAVING SUM(i.Total) > 700 AND AVG(i.Total) > 5.5
```


## Exercise 9 – Subtle mistake awareness

**Task:** Write **two** queries that answer the same business question, but one uses the wrong clause.

**Business question:**  
Among customers in the **USA** or **Canada**, which of those countries have more than 5 customers?

### Instructions
1. First write the **correct** query (filter countries with WHERE, then HAVING on the count).
2. Then write an **incorrect** version that tries to put the country filter inside HAVING.
3. Observe that the incorrect version still “works” in this particular case but is conceptually wrong and can become incorrect with different data.


**Hints – Correct version**
```sql
WHERE Country IN ('USA', 'Canada')
GROUP BY Country
HAVING COUNT(*) > 5
```

**Incorrect (but tempting) version**
```sql
GROUP BY Country
HAVING Country IN ('USA', 'Canada') AND COUNT(*) > 5
```
The second form filters after grouping. It happens to give the same answer here, but it forces the database to build groups for every country first — less efficient and easy to misuse when the filter should eliminate rows earlier.


## Exercise 10 – Challenge: Layered filters

**Question:**  
For invoices issued in **2011, 2012 or 2013**, show billing countries that:
- have **at least 8** invoices in those years, **and**
- whose **maximum** invoice in those years is **greater than 10**, **and**
- whose **average** invoice is **at least 5.5**

Return country, invoice count, max total, and average total.  
Order by average total descending.


**Hints**
- `WHERE` for the year filter
- `GROUP BY BillingCountry`
- `HAVING COUNT(*) >= 8 AND MAX(Total) > 10 AND AVG(Total) >= 5.5`
